# ViT5: 3 cấu hình trên toàn bộ validation

Không dùng prefix. Cả 3 cấu hình đều `beam=4`, `max_new_tokens=128`; chỉ thay `length_penalty`: 0.8, 1.0, 1.1. Prediction xuất đúng 6 trường chuẩn.

In [1]:
%pip install -q "transformers>=4.41,<5" sentencepiece accelerate rouge-score tqdm ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import gc, json, random, re, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from rouge_score import rouge_scorer, scoring
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
MODEL_NAME="VietAI/vit5-base-vietnews-summarization"
SYSTEM="vit5_base"
MAX_SOURCE_LENGTH=1024
BATCH_SIZE=4
EVAL_LIMIT=None  # Chay toan bo validation
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
OUT=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
print('Device:',DEVICE,torch.cuda.get_device_name(0) if DEVICE=='cuda' else '')

Device: cuda Tesla T4


In [3]:
CONFIGS=[
    {"num_beams":4,"length_penalty":0.8,"max_new_tokens":128,"min_new_tokens":0,"no_repeat_ngram_size":3},
    {"num_beams":4,"length_penalty":1.0,"max_new_tokens":128,"min_new_tokens":0,"no_repeat_ngram_size":3},
    {"num_beams":4,"length_penalty":1.1,"max_new_tokens":128,"min_new_tokens":0,"no_repeat_ngram_size":3},
]
assert len(CONFIGS)==3
print(CONFIGS)

[{'num_beams': 4, 'length_penalty': 0.8, 'max_new_tokens': 128, 'min_new_tokens': 0, 'no_repeat_ngram_size': 3}, {'num_beams': 4, 'length_penalty': 1.0, 'max_new_tokens': 128, 'min_new_tokens': 0, 'no_repeat_ngram_size': 3}, {'num_beams': 4, 'length_penalty': 1.1, 'max_new_tokens': 128, 'min_new_tokens': 0, 'no_repeat_ngram_size': 3}]


In [4]:
def find_file(name):
    roots=[Path('/kaggle/input/newdatavit5/data'),Path('/kaggle/input'),Path.cwd()]
    found=[]
    for root in roots:
        if root.exists():
            direct=root/name
            if direct.is_file(): found.append(direct.resolve())
            found.extend(p.resolve() for p in root.rglob(name) if p.is_file())
    found=list(dict.fromkeys(found))
    if not found: raise FileNotFoundError(f'Khong tim thay {name}')
    print(name,'->',found[0]); return found[0]

def clean_text(x):
    x=re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]','',str(x))
    return re.sub(r'\s+',' ',x).strip()

def load_jsonl(path):
    rows=[]
    with path.open(encoding='utf-8-sig') as f:
        for n,line in enumerate(f,1):
            if line.strip():
                try: rows.append(json.loads(line))
                except json.JSONDecodeError as e: raise ValueError(f'JSON loi dong {n}: {e}') from e
    df=pd.DataFrame(rows)
    required={'id','source','reference'}
    if not required<=set(df.columns): raise ValueError(f'Thieu cot: {required-set(df.columns)}')
    df=df.dropna(subset=list(required)).copy()
    for c in required: df[c]=df[c].map(clean_text)
    df=df[(df.id!='')&(df.source!='')&(df.reference!='')].reset_index(drop=True)
    assert df.id.is_unique and len(df)>0
    return df

validation_df=load_jsonl(find_file('validation_select.jsonl'))
work_df=validation_df if EVAL_LIMIT is None else validation_df.head(EVAL_LIMIT).copy()
print('So mau se chay:',len(work_df)); display(work_df[['id','source','reference']].head(2))

validation_select.jsonl -> /kaggle/input/datasets/tanluu/newdatavit5/data/validation_select.jsonl
So mau se chay: 2000


,id,source,reference
0,validation_000007,Đối tượng bị khởi tố là Nông Văn Phú (SN 1985)...,"Lạng Sơn - Ngày 23.3, Cơ quan Cảnh sát điều tr..."
1,validation_000014,"Bước vào cuộc tiếp đón Benfica, HLV Jose Mouri...",M.U đã không gặp nhiều khó khăn trong việc đả ...


In [5]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME,use_fast=False)
kwargs={"dtype":torch.float16} if DEVICE=='cuda' else {}
model=AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME,trust_remote_code=False,**kwargs).to(DEVICE)
model.eval(); print('Loaded:',MODEL_NAME)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/904M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/904M [00:00<?, ?B/s]

Loaded: VietAI/vit5-base-vietnews-summarization


In [6]:
def config_id(c):
    return f"beam{c['num_beams']}_lp{c['length_penalty']:g}_max{c['max_new_tokens']}_nr{c['no_repeat_ngram_size']}"

@torch.inference_mode()
def summarize(texts,cfg):
    preds,errors=[],[]
    for start in tqdm(range(0,len(texts),BATCH_SIZE),desc=config_id(cfg)):
        raw=texts[start:start+BATCH_SIZE]
        batch=[clean_text(x) for x in raw]
        try:
            enc=tokenizer(batch,max_length=MAX_SOURCE_LENGTH,truncation=True,padding=True,return_tensors='pt').to(DEVICE)
            ids=model.generate(**enc,max_new_tokens=cfg['max_new_tokens'],min_new_tokens=cfg['min_new_tokens'],num_beams=cfg['num_beams'],length_penalty=cfg['length_penalty'],no_repeat_ngram_size=cfg['no_repeat_ngram_size'],early_stopping=True,do_sample=False)
            out=[clean_text(x) for x in tokenizer.batch_decode(ids,skip_special_tokens=True)]
            for x in out:
                preds.append(x); errors.append(None if x else 'Empty prediction')
        except Exception as e:
            msg=f'{type(e).__name__}: {e}'
            preds.extend(['']*len(raw)); errors.extend([msg]*len(raw))
            if DEVICE=='cuda': torch.cuda.empty_cache()
    assert len(preds)==len(texts)==len(errors)
    return preds,errors

def make_prediction(ids,preds,errors,cfg):
    df=pd.DataFrame({'id':ids,'system':SYSTEM,'config_id':config_id(cfg),'prediction':preds,'status':['ok' if p and e is None else 'error' for p,e in zip(preds,errors)],'error':errors})
    expected=['id','system','config_id','prediction','status','error']
    assert list(df.columns)==expected and df.id.is_unique
    assert df.status.isin(['ok','error']).all()
    assert df.loc[df.status=='ok','prediction'].str.len().gt(0).all()
    return df

def score(preds,refs):
    s=rouge_scorer.RougeScorer(['rouge1','rouge2','rougeLsum'],use_stemmer=False); a=scoring.BootstrapAggregator(); n=0
    for p,r in zip(preds,refs):
        if p: a.add_scores(s.score(r,p)); n+=1
    if not n: return {'rouge1':np.nan,'rouge2':np.nan,'rougeLsum':np.nan,'n_scored':0}
    z=a.aggregate(); return {k:round(z[k].mid.fmeasure*100,4) for k in z}|{'n_scored':n}

def save_prediction(df,stem):
    csv_path=OUT/f'{stem}.csv'; jsonl_path=OUT/f'{stem}.jsonl'
    df.to_csv(csv_path,index=False,encoding='utf-8-sig')
    with jsonl_path.open('w',encoding='utf-8') as f:
        for rec in df.to_dict('records'):
            if pd.isna(rec['error']): rec['error']=None
            f.write(json.dumps(rec,ensure_ascii=False)+'\n')
    chk=pd.read_csv(csv_path,encoding='utf-8-sig',keep_default_na=False)
    assert list(chk.columns)==['id','system','config_id','prediction','status','error']
    assert len(chk)==len(df) and chk.id.is_unique
    print('Saved:',csv_path); print('Saved:',jsonl_path)
    return csv_path,jsonl_path

In [7]:
metrics=[]; reviews=[]
for cfg in CONFIGS:
    t=time.time(); preds,errors=summarize(work_df.source.tolist(),cfg)
    pred_df=make_prediction(work_df.id.tolist(),preds,errors,cfg)
    save_prediction(pred_df,f"validation_predictions_{config_id(cfg)}")
    m=score(preds,work_df.reference.tolist())
    m.update({'config_id':config_id(cfg),'n_samples':len(work_df),'n_errors':int((pred_df.status=='error').sum()),'avg_words':round(np.mean([len(x.split()) for x in preds if x]),2),'seconds':round(time.time()-t,1)})
    metrics.append(m)
    review=work_df[['id','reference']].copy(); review['config_id']=config_id(cfg); review['prediction']=preds; reviews.append(review)
    gc.collect()
    if DEVICE=='cuda': torch.cuda.empty_cache()
metrics_df=pd.DataFrame(metrics).sort_values(['rougeLsum','rouge2'],ascending=False).reset_index(drop=True)
metrics_df.to_csv(OUT/'validation_config_scores.csv',index=False,encoding='utf-8-sig')
pd.concat(reviews,ignore_index=True).to_csv(OUT/'validation_config_analysis.csv',index=False,encoding='utf-8-sig')
display(metrics_df)

beam4_lp0.8_max128_nr3:   0%|          | 0/500 [00:00<?, ?it/s]

Saved: /kaggle/working/validation_predictions_beam4_lp0.8_max128_nr3.csv
Saved: /kaggle/working/validation_predictions_beam4_lp0.8_max128_nr3.jsonl


beam4_lp1_max128_nr3:   0%|          | 0/500 [00:00<?, ?it/s]

Saved: /kaggle/working/validation_predictions_beam4_lp1_max128_nr3.csv
Saved: /kaggle/working/validation_predictions_beam4_lp1_max128_nr3.jsonl


beam4_lp1.1_max128_nr3:   0%|          | 0/500 [00:00<?, ?it/s]

Saved: /kaggle/working/validation_predictions_beam4_lp1.1_max128_nr3.csv
Saved: /kaggle/working/validation_predictions_beam4_lp1.1_max128_nr3.jsonl


,rouge1,rouge2,rougeLsum,n_scored,config_id,n_samples,n_errors,avg_words,seconds
0,58.4208,28.6045,37.7829,2000,beam4_lp1_max128_nr3,2000,0,36.72,895.7
1,58.5116,28.6310,37.7672,2000,beam4_lp1.1_max128_nr3,2000,0,36.89,896.0
2,58.1494,28.5303,37.6970,2000,beam4_lp0.8_max128_nr3,2000,0,36.18,905.1
